# Activity 002/003: Sequence Alignment

## 1. Install the Python library to read the FASTA files

In [ ]:
%pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.4 MB/s eta 0:00:00


## 2. Functions from previous activities

In [ ]:
from Bio import SeqIO
from google.colab import drive
drive.mount('/content/drive')

def get_fasta_info(file_path):
    # Open the FASTA file
    with open(file_path, "r", encoding="utf-8") as fasta_file:
        record = SeqIO.read(fasta_file, "fasta")  # read in the FASTA records
        print(f"Description: {record.description}")
        print(f"Sequence: {record.seq[:20]}...")  # Print first 20 bases
        print(f"Length: {len(record)}\n")

def get_dna_seq(file_path):
    # Open the FASTA file
    with open(file_path, "r", encoding="utf-8") as fasta_file:
        record = SeqIO.read(fasta_file, "fasta")
        return record.seq

base_path = "/content/drive/MyDrive/.../"   # REPLACE with your path
actino_fasta_path = f"{base_path}data/actinobacteria_dna.fasta"
plasmodium_fasta_path = f"{base_path}data/plasmodium_falciparum_dna.fasta"

# Grab the DNA sequences
actino_dna = get_dna_seq(actino_fasta_path)
plasmodium_dna = get_dna_seq(plasmodium_fasta_path)

Mounted at /content/drive


## 3. Hamming Distance
First, grab the first 20 characters of the two DNA sequences and print them out.

In [ ]:
# TO DO: grab the first 20 characters of each DNA to compare
# seq_1 =
# seq_2 =
print(seq_1)
print(seq_2)

AAGCACGACCTGGGCAACCT
AAGCTTTTGGTATCTCGTAA


Next, we define a function that takes in two sequences and calculates the Hamming Distance. It also recreates the strings of the DNA sequences with red font for where the characters differ.

In [ ]:
def get_hamming_distance(seq_1, seq_2):
    hamming_distance = 0
    seq_1_print = []
    seq_2_print = []
    red_font = "\033[31m"
    reset_font = "\033[0m"

    for char_1, char_2 in zip(seq_1, seq_2):
        if char_1 != char_2:
            # TO DO:
            # Increase the Hamming Distance

            # Rewrite both sequences with red font where the characters differ
            seq_1_print.append(f"{red_font}{char_1}{reset_font}")
            seq_2_print.append(f"{red_font}{char_2}{reset_font}")

        else:
            seq_1_print.append(char_1)
            seq_2_print.append(char_2)

    seq_1_print = "".join(seq_1_print)
    seq_2_print = "".join(seq_2_print)

    return hamming_distance, seq_1_print, seq_2_print

# Get the Hamming Distance for the first 20 characters of actino and plasmodium DNA
hamming_distance, seq_1_print, seq_2_print = get_hamming_distance(seq_1, seq_2)
print(f"Hamming Distance: {hamming_distance}")
print(f"{seq_1_print}")
print(f"{seq_2_print}")

Hamming Distance: 15
AAGCACGACCTGGGCAACCT
AAGCTTTTGGTATCTCGTAA


## 4. Global Sequence Alignment
We define the following functions to implement the Global Sequence Alignment algorithm. You may recognize the scoring function `f()`.

The following code is optional to understand but feel free to look more into it:
- The scoring function `f()` gets used inside `get_dp_matrices()` to build and save the optimal scores at each step
- `backtrack()` goes through the optimal score matrix and finds the path that gives the optimal score, resulting in the optimal alignment of sequences.
- `get_alignment()` is a wrapper function that essentially just calls `backtrack()` and serves as an easy interface for users. This is what you will call to find the optimal alignment of two sequences.

In [ ]:
import numpy as np

# Pointer Codes:
#     0: Diagonal,
#     1: Up,
#     2: Left,
#     3: None (only used for the top-left corner that has no parents)
#

GAP = "-"

def f(char_1: str, char_2: str) -> int:
    """
    Calculates score according to the given score scheme and linear gap penalty
    :param char_1: a character in the sequence
    :param char_2: a character in the other sequence
    :return: score for two characters
    """
    if char_1 == GAP or char_2 == GAP:
        return GAP_PENALTY
    elif char_1 == char_2:
        return MATCH
    else:
        return MISMATCH

def get_dp_matrices(S: str, T: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Calculates the scoring matrix according to the global
    alignment recurrence relation.
    :param S: sequence 1
    :param T: sequence 2
    :return: scoring matrix, pointer matrix
    """
    m = len(S)
    n = len(T)
    scores = np.zeros((m + 1, n + 1), dtype=int)
    pointers = np.zeros((m + 1, n + 1), dtype=int)

    # set top-left corner of pointers matrix to
    # having no further pointers
    pointers[0, 0] = 3

    # setting negative decrements for each '-' that
    # is matched with a character in S or T
    for i in range(1, m+1):
        scores[i,0] = scores[i-1, 0] + f(S[i-1], GAP)
        pointers[i,0] = 1

    for i in range(1, n+1):
        scores[0, i] = scores[0, i-1] + f(GAP, T[i-1])
        pointers[0, i] = 2

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # calculate all possible scores
            score_options = np.array([
                scores[i - 1, j - 1] + f(S[i - 1], T[j - 1]), # diagonal
                scores[i - 1, j] + f(S[i - 1], GAP),          # up
                scores[i, j - 1] + f(GAP, T[j - 1]),          # left
            ])

            # find the largest score
            scores[i, j] = np.max(score_options)

            # find and set the parent of largest score
            pointers[i, j] = int(np.argmax(score_options))

    return scores, pointers

def backtrack(
    i: int,
    j: int,
    pointers: np.ndarray,
    S: str,
    T: str,
) -> tuple[str, str]:
    """
    Backtracks through the maximum scores in the scoring matrix
    and stopping when all characters have been aligned
    :param i: row index of current cell in scoring matrix
    :param j: column index of current cell in scoring matrix
    :param pointers: parent pointer matrix for alignment scores
    :param S: sequence 1
    :param T: sequence 2
    :return: the optimal alignment of S with a subset of T
    """

    # Initialize the S and T alignments as lists of strings
    # The characters from S and T will be added to these lists
    # from right to left
    S_aligned = []
    T_aligned = []

    # Recursive Case
    # Make your from the bottom-right corner of the pointers matrix
    # up to the top-left corner
    while pointers[i, j] != 3:
        if pointers[i, j] == 0:
            S_aligned.append(S[i - 1])
            T_aligned.append(T[j - 1])
            i -= 1
            j -= 1
        elif pointers[i, j] == 1:
            S_aligned.append(S[i - 1])
            T_aligned.append(GAP)
            i -= 1
        elif pointers[i, j] == 2:
            S_aligned.append(GAP)
            T_aligned.append(T[j - 1])
            j -= 1

    # Reverse the strings
    S_aligned = "".join(reversed(S_aligned))
    T_aligned = "".join(reversed(T_aligned))

    return S_aligned, T_aligned

def get_alignment(S: str, T: str) -> tuple[str, str, int]:
    """
    Calls the backtrack function to start at the bottom-right corner of the scoring matrix
    and then reverses the returned alignment strings to be in the original order
    :param S: sequence 1
    :param T: sequence 2
    :return: optimal alignment of S and T and the alignment score
    """
    m = len(S)
    n = len(T)

    # run the alignment score calculations
    scores, pointers = get_dp_matrices(S, T)

    # max score is at the bottom-right corner of the scores matrix
    max_score = int(scores[m, n])

    # call the backtrack function starting at the bottom-right corner
    S_aligned, T_aligned = backtrack(m, n, pointers, S, T)

    return S_aligned, T_aligned, max_score

## Running Global Alignment
Here we will test out the global alignment algorithm with different scoring schemes to see how it changes the optimal alignment.
Feel free to change S and T and the scoring scheme to see different results!

In [ ]:
# Scoring Scheme:
MATCH = 1
MISMATCH = -1
GAP_PENALTY = -1

# Sequences
S = "ACGGT"
T = "AGGCT"

# run the backtracking to get the global alignment
S_aligned, T_aligned, max_score = get_alignment(S, T)

# output results
print(f"Global Alignment Score: {max_score}")
print(f"{S_aligned}")
print(f"{T_aligned}")

Global Alignment Score: 2
ACGG-T
A-GGCT


In [ ]:
# TO DO: use a different scoring scheme
MATCH =
MISMATCH =
GAP_PENALTY =   # Hint: set a larger gap penalty

S = "ACGGT"
T = "AGGCT"

# run the backtracking to get the global alignment
S_aligned, T_aligned, max_score = get_alignment(S, T)

# output results
print(f"Global Alignment Score: {max_score}")
print(f"{S_aligned}")
print(f"{T_aligned}")

Global Alignment Score: 1
ACGGT
AGGCT


# 5. Experiment
As we can see above , when we set the penalty for gaps to be larger, this changed how often gaps were used in the global alignment.

Feel free to copy the code above and change the sequences and scoring scheme to experiment.